---
authors:
  - edesz
date: 2026-04-08
---

# Model Interpretability

In [ ]:
import os
from datetime import datetime
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
import shap
from dotenv import load_dotenv

In [ ]:
_ = shap.initjs()

In [ ]:
PROJ_ROOT = Path.cwd().parent

In [ ]:
assert load_dotenv(dotenv_path=PROJ_ROOT.parent / ".env")

In [ ]:
import cc_churn.explanation as shexp
import cc_churn.visualization as vzu
import r2.io_utils as r2io
from utils.df_utils import show_df

## About

Provide model interpredation for predicted churners and non-churners, among all available customers' data.

## User Inputs

In [ ]:
# R2 data bucket details
# # name of train data key (file) in private R2 bucket
r2_key_train = "train_data.parquet.gzip"
# # name of validation data key (file) in private R2 bucket
r2_key_val = "validation_data.parquet.gzip"
# # name of test data key (file) in private R2 bucket
r2_key_test = "test_data.parquet.gzip"

# # datatypes for categorical and ordinal columns
# dtypes_ordinals = {
#     "gender": "string[pyarrow]",
#     "income_category": "string[pyarrow]",
#     "education_level": "string[pyarrow]",
# }
# dtypes_categoricals = {
#     "marital_status": "category",
#     "card_category": "category",
# }

# trained model prefix
key_prefix = "2026-04-13/best_model__"

# predictions prefix
r2_key_pred = "2026-04-13/all_predictions__"
cols_predictions = ["clientnum", "y_pred", "y_pred_proba"]

r2_key_pred_bm_savings = (
    "2026-04-13/at_risk_customers_with_business_metrics_savings__"
)

label = "is_churned"

In [ ]:
account_id = os.getenv("ACCOUNT_ID")
access_key_id = os.getenv("ACCESS_KEY_ID_USER2")
secret_access_key = os.getenv("SECRET_ACCESS_KEY_USER2")
bucket_name = os.getenv("BUCKET_NAME")

s3_client = boto3.client(
    "s3",
    endpoint_url=f"https://{account_id}.r2.cloudflarestorage.com",
    aws_access_key_id=access_key_id,
    aws_secret_access_key=secret_access_key,
    region_name="auto",
)

## Load Data

### Trained Model

Load pipeline

In [ ]:
%%time
pipe_best = r2io.joblib_load_from_r2(s3_client, bucket_name, key_prefix)
pipe_best

Extract transformer and preprocessor as separate pipeline and model

In [ ]:
pipe_transf_pre = pipe_best.estimator_[:-1]
model = pipe_best.estimator_.named_steps["clf"]

Get the best features, that were also preprocessed by the best pipeline (`pipe_best`) above

In [ ]:
best_features = (
    pipe_transf_pre.named_steps["pre"]
    .named_transformers_["num"]
    .feature_names_in_.tolist()
)

### Predictions

Load scenarios (using estimated savings) from predictions

In [ ]:
%%time
df_scenarios = r2io.pandas_read_filtered_parquets_r2(
    s3_client,
    bucket_name,
    r2_key_pred_bm_savings,
    ["clientnum", "value_tier", "risk_level", "scenario"],
)
_ = show_df(df_scenarios)
with pd.option_context("display.max_columns", None):
    display(df_scenarios.head(1))

Load predictions for each customer in all available data

In [ ]:
%%time
df = (
    pd.concat(
        [
            r2io.pandas_read_parquet_r2(s3_client, bucket_name, k)
            for k in [r2_key_train, r2_key_val, r2_key_test]
        ]
    ).merge(
        r2io.pandas_read_filtered_parquets_r2(
            s3_client, bucket_name, r2_key_pred, cols_predictions
        ),
        on="clientnum",
        how="left",
    )
)
_ = show_df(df)
with pd.option_context("display.max_columns", None):
    display(df.head(1))

## Separate Features from Target

In [ ]:
X = df.drop(columns=[label])
y = df[label]

Get the features that were not used in training for the best pipeline above

In [ ]:
features_non_preprocessed = list(set(list(X)) - set(best_features))

## Transform Data

### Non-Numerical Features with Rare Categories

This custom category combining transformer is applied below to all data

In [ ]:
%%time
# train custom transformer and preprocessor on data
_ = pipe_transf_pre.fit(X)

# use trained pipeline to get column names after transformation & preprocessing
columns_transformed = list(
    pipe_transf_pre.get_feature_names_out(input_features=list(X)).tolist()
)

# apply trained transformer to perform category grouping on all data
X_transformed_preprocessed = pd.DataFrame(
    pipe_transf_pre.transform(X),
    columns=columns_transformed,
    index=X.index,
)
_ = show_df(X_transformed_preprocessed)

### Pre-Process Numerical Features

Combine non-pre processed features, pre-processed numerical features and the true label for all customers

In [ ]:
%%time
df_transf_pre = pd.concat(
    [X[features_non_preprocessed], X_transformed_preprocessed, y.rename("y")],
    axis=1,
)[list(X)+['y']]

In [ ]:
%%time
df_scenarios_transf_pre = df_transf_pre.merge(
    df_scenarios[["clientnum", "risk_level", "value_tier", "scenario"]],
    on=["clientnum"],
    how="inner",
)
with pd.option_context("display.max_columns", None):
    display(df_scenarios_transf_pre.head(1))

## Model Interpretability

### Predicted Churners

Get SHAP values (`shap_values`) and SHAP explanation object (`shap_explainer_values_*`) for the predicted churners

In [ ]:
%%time
_, shap_explainer_values_pc = shexp.get_shap_values(
    model, df_transf_pre.query("y_pred == 1")[best_features]
)

Plot SHAP summary subplots consisting of

1. (LHS) [beeswarm plot](https://shap.readthedocs.io/en/latest/example_notebooks/api_examples/plots/beeswarm.html#A-simple-beeswarm-summary-plot) that shows how the most important features impact the model's predictions for churners
2. (RHS) [bar plot](https://shap.readthedocs.io/en/latest/example_notebooks/api_examples/plots/bar.html#Global-bar-plot) showing the global feature importance, across all predicted churners

In [ ]:
%%time
vzu.plot_shap_summary_plots(
    shap_explainer_values_pc,
    y_axis_annot_offset=0,
    axis_label_fontsize=18,
    y_axis_annot_fontsize=16,
    wspace=0.75,
    fig_size=(16, 10),
)

The width of the bars in the bar plot (left) indicates the importance of the features to the best model's predictions. Bar width is calculated as the average of the absolute value of the individual SHAP values for all predicted churners (customers predicted to cancel their credit card services at the bank).

The beeswarm plot (right) indicates the directionality of the individual contribution to model predictions by each feature: red indicates a higher contribution while blue indicates a lower contribution ([link](https://doi.org/10.1038/s41598-022-07881-2)). It indicates how the most important features impact the model's predictions for churners. The order of features is based on the average of the absolute value of SHAP values for each feature and so it places more emphasis on broad average impact across all predicted churners.

### Predicted Non-Churners

Get the global SHAP values and SHAP explanation object for the predicted non-churners

In [ ]:
%%time
_, shap_explainer_values_pnc = shexp.get_shap_values(
    model, df_transf_pre.query("y_pred == 0")[best_features]
)

Plot SHAP summary subplots

In [ ]:
%%time
vzu.plot_shap_summary_plots(
    shap_explainer_values_pnc,
    y_axis_annot_offset=0,
    axis_label_fontsize=18,
    y_axis_annot_fontsize=16,
    wspace=0.75,
    fig_size=(16, 10),
)

In [ ]:
%%time
_, shap_explainer_values_sc1 = shexp.get_shap_values(
    model, df_scenarios_transf_pre.query("scenario == 1")[best_features]
)

In [ ]:
%%time
vzu.plot_shap_summary_plots(
    shap_explainer_values_sc1,
    y_axis_annot_offset=0,
    axis_label_fontsize=18,
    y_axis_annot_fontsize=16,
    wspace=0.75,
    fig_size=(16, 10),
)

In [ ]:
%%time
_, shap_explainer_values_sc2 = shexp.get_shap_values(
    model, df_scenarios_transf_pre.query("scenario == 2")[best_features]
)

In [ ]:
%%time
vzu.plot_shap_summary_plots(
    shap_explainer_values_sc2,
    y_axis_annot_offset=0,
    axis_label_fontsize=18,
    y_axis_annot_fontsize=16,
    wspace=0.75,
    fig_size=(16, 10),
)

## Conclusion